In [1]:
# Kármán momentum integral with the Majdalani-Xuan quartic. Written by Claude Code (claude-opus-5-5), 2026-10-27;
# see AI-USE.md section 2.
# Profile: the Majdalani-Xuan quartic, g = (5 xi - 3 xi^3 + xi^4)/3, from White and Majdalani (2022), Table 4-1, p.158,
# the author's choice of guessed profile for the momentum integral.
# Derivation of ALPHA, BETA, delta, theta, C_f and avg C_f: Benjamin Moors, the author's own; checked by Claude Code.
import numpy as np
import pandas as pd
from scipy.integrate import quad

import sys
from pathlib import Path

# The repository root, so that the src package imports from this notebook's folder (moved 2026-10-28)
sys.path.insert(0, str(Path.cwd().parents[2]))
from src.friction.von_karman import von_karman as vk
from src.friction.blasius_shooting import skin_and_shear as ss

pd.set_option("display.precision", 12)

# Salt water at 20 degC, as SWD uses: rho = 1025 kg/m^3, mu = 0.00107 Pa s
U, NU = 2.0, 0.00107/1025

In [2]:
# ALPHA = theta/delta = integral of g(1 - g) over the layer, against the author's 379/2835
alpha_quad, alpha_quad_error = quad(lambda xi: vk.profile(xi)*(1 - vk.profile(xi)), 0.0, 1.0)

# BETA = g'(0) by the 3-point wall stencil at h = 0.02 and 0.01, against the author's 5/3
slopes = [ss.wall_slope(vk.profile(np.arange(0.0, 2.0 + h/2, h)), h) for h in (0.02, 0.01)]

pd.DataFrame({"author": [379/2835, 5/3],
              "computed": [alpha_quad, slopes[1]],
              "computed - author": [alpha_quad - 379/2835, slopes[1] - 5/3],
              "check bound": [alpha_quad_error, abs(slopes[0] - slopes[1])]},
             index=["ALPHA, quad", "BETA, wall_slope at h = 0.01"])
# check bound: quad's own error estimate for ALPHA; the change from h to h/2 for BETA

,author,computed,computed - author,check bound
"ALPHA, quad",0.133686067019,0.133686067019,-5.551115123126e-17,1.484213496765e-15
"BETA, wall_slope at h = 0.01",1.666666666667,1.666864666667,1.979999999997e-04,5.860000000004e-04


In [3]:
# RK4 march of d(delta)/dx = nu*BETA/(U*ALPHA*delta) against the closed form, on the author's grid: x0 = 0.01 m,
# x_end = 0.5 m, h = 0.49/64 m, then h/2 and h/4 (AI-USE.md section 4, 2026-10-27).
# Limit: the march starts from the closed-form delta(x0), because the ODE is singular at delta(0) = 0; only x0 onward
# is numerical. theta = ALPHA*delta, so its relative error is delta's.
X0, X_END = 0.01, 0.5
H = (X_END - X0)/64
runs = [vk.march_delta(X0, X_END, U, NU, H/2**k) for k in range(3)]

table = pd.DataFrame({
    "h (m)": [H/2**k for k in range(3)],
    "max |delta_RK4 - delta|": [np.max(np.abs(d - vk.delta(x, U, NU))) for x, d in runs],
    "max |theta_RK4 - theta|": [np.max(np.abs(vk.ALPHA*d - vk.theta(x, U, NU))) for x, d in runs]})

# Oberkampf's p_obs on the shared (coarse) points, no exact solution needed; the author's tolerance is 0.1 on 4
change = [np.max(np.abs(a[1] - b[1][::2])) for a, b in zip(runs, runs[1:])]
p_obs = np.log2(change[0]/change[1])
print(f"p_obs = {p_obs:.4f} (|p_obs - 4| <= 0.1: {abs(p_obs - 4) <= 0.1}); "
      f"error at h/4 within the h/2 to h/4 change: {table.iloc[2, 1] <= change[1]}")
table

p_obs = 4.0694 (|p_obs - 4| <= 0.1: True); error at h/4 within the h/2 to h/4 change: True


,h (m),max |delta_RK4 - delta|,max |theta_RK4 - theta|
0,0.0076562500,0.000000020546,0.000000002747
1,0.0038281250,0.000000001223,0.000000000163
2,0.0019140625,0.000000000072,0.000000000010


In [4]:
# The quartic against its own Table 4-1 row (p.158, printed to 3 decimals). The same profile, so they should
# agree to half the last printed digit.
eta_star, _ = quad(lambda xi: 1 - vk.profile(xi), 0.0, 1.0)  # delta*/delta
x = 0.1
Re_x = U*x/NU
delta_coefficient = vk.delta(x, U, NU)*np.sqrt(Re_x)/x
row = pd.DataFrame({
    "computed": [eta_star, vk.ALPHA, eta_star/vk.ALPHA, delta_coefficient, vk.C_f(Re_x)*np.sqrt(Re_x),
                 eta_star*delta_coefficient],
    "Table 4-1": [0.350, 0.134, 2.618, 4.993, 0.668, 1.748]},
    index=["eta* = delta*/delta", "theta* = theta/delta", "H", "(delta/x) sqrt(Re_x)", "C_f sqrt(Re_x)",
           "(delta*/x) sqrt(Re_x)"])
row["difference"] = row["computed"] - row["Table 4-1"]
row["within 0.0005"] = row["difference"].abs() <= 0.5e-3
display(row)

# Gap from Blasius (the RK4 shooting f''(0)): reported, not asserted (AI-USE.md section 4, 2026-10-26).
# Table 4-1 prints the quartic's C_f gap from its Blasius row (0.664) as 0.53 %; the shooting f''(0) is unrounded.
s_star, _ = ss.shoot()
gap = pd.DataFrame({"Karman quartic": [vk.C_f(Re_x)*np.sqrt(Re_x), vk.avg_C_f(Re_x)*np.sqrt(Re_x)],
                    "Blasius shooting": [2*s_star, 4*s_star]},
                   index=["C_f sqrt(Re_x)", "avg C_f sqrt(Re_L)"])
gap["gap from Blasius (%)"] = 100*(gap["Karman quartic"]/gap["Blasius shooting"] - 1)
gap

,computed,Table 4-1,difference,within 0.0005
eta* = delta*/delta,0.350000000000,0.350,0.000000000000,True
theta* = theta/delta,0.133686067019,0.134,-0.000313932981,True
H,2.618073878628,2.618,0.000073878628,True
(delta/x) sqrt(Re_x),4.993399337056,4.993,0.000399337056,True
C_f sqrt(Re_x),0.667547918428,0.668,-0.000452081572,True
(delta*/x) sqrt(Re_x),1.747689767970,1.748,-0.000310232030,True


,Karman quartic,Blasius shooting,gap from Blasius (%)
C_f sqrt(Re_x),0.667547918428,0.664114679009,0.516964844751
avg C_f sqrt(Re_L),1.335095836857,1.328229358018,0.516964844751
